Finora in IR classico abbiamo ragionato soprattutto sul contenuto dei documenti: parole presenti, tf-idf, BM25, similarità tra query e documento etc...

Con la **Link Analysis** si guarda oltre: non ci si limita più solo a quello che è effettivamente scritto nella pagina, ma anche a **come quella pagina è collegata ad altre pagine** (e quindi se è **autorevole**).  
Infatti nel Web ogni pagina può contenere link verso altre pagine. Ognuno di questi link contiene un'**ancora** (anchor) che è un testo che descrive la pagina a cui punta. Una pagina molto linkata (soprattutto se da pagine autorevoli) potrebbe essere considerata più importante di una pagina isolata.

Intuitivamente possianmo distinguere i nodi (dove ogni nodo rappresenta una pagina) tra **Good**, **Bad** e **Unknown**, e in base ai link che li collegano possiamo inferire la natura dei nodi Unknown in quanto difficilmente un nodo buono punta a un nodo cattivo.

<p align="center">
    <img src="img/111.png" width="45%" />
    <img src="img/222.png" width="45%" />
</p>

Ovviamente questa logica deve essere applicata con le dovute precauzioni, in quanto nella realtà una pagina buona potrebbe linkare una pagina cattiva ad esempio per criticarla, in questo senso nella realtà la classificazione non è rigida ma probabilistica.

Nel Web inoltre la distribuzione dei link non è uniforme e segue una **distribuzione potenza** (**Power Law**) per cui pochissime pagine sono enormemente popolari, mentre moltissime sono poco linkate.  
In questo senso era un problema lo spam: una pagina poco autorevole poteva apparire come molto linkata creando molti link fasulli da pagine create ad hoc.

<img src="img/333.png" width="400" />

Chiaramente link analysis è applicabile a molti domini oltre al Web, tra cui ad esempio reti sociali, sistemi di email, analisi delle frodi, reti telefoniche etc.. (es. se una persona ha come molti amici persone che spendono molto, è probabile che anche lei spenda molto).

Tornando però al mondo dell'IR, le applicazioni principali della Link Analysis sono le seguenti:
- **Scoring e Ranking**: i link possono aiutare a decidere quali documenti mostrare prima nei risultati di ricerca (es. se due pagine contengono le stesse parole, quella più linkata potrebbe essere più autorevole e quindi mostrata prima).
- **Link-based Clustering**: i link possono essere usati per individuare comunità tematiche (se molte pagine si linkano tra loro, è probabile che trattino argomenti simili).
- **Crawling**: ogni giorno nascono migliaia di nuovi siti. I link possono essere usati per scoprire nuove pagine da indicizzare (es. se una pagina è molto linkata, è probabile che sia importante e quindi valga la pena indicizzarla presto).

### Formalizzazione del problema e assunzioni
Si formalizza il Web come un grafo diretto, dove i nodi rappresentano le pagine e gli archi rappresentano gli hyperlink tra pagine.

Si introducono due ipotesi fondamentali:
1. **Un hyperlink conferisce autorità, se presente in una pagina reputata autorevole**, questa è l'intuizione che porterà a PageRank.
2. **L'anchor text descrive la pagina di destinazione**

Quando un documento D viene indicizzato nell'inverted index, sappiamo che nell'indice vengono inserite le parole presenti in D. Con l'anchor text si fa qualcosa di più: oltre alle parole di D, vengono anche indicizzate le parole presenti negli anchor text dei link che puntano a D, con un certo peso. In questo modo la pagina può essere recuperata anche con query che non contengono esattamente le parole presenti in D, ma parole presumibilmente correlate (se erano negli anchor text di link che puntano a D).

In questo senso è utile salvare nell'indice anche parole attorno all'anchor text, sempre perché potrebbero essere correlate alla pagina di destinazione (es."More information about IBM products can be found here"; here sarebbe l'anchor text ma non dà informazioni.. se prendo però le parole prima allora molto meglio).

**Problema**: spam e "miserable failure", si fa riferimento al fenomeno per cui alcune pagine possono usare anchor text fallaci verso una pagina per far sì che i motori di ricerca siano portati a restituire quella pagina anche per query che non c'entrano nulla.

Per risolvere il problema la strategia principale è come **pesare l'anchor text in base all'autorità della pagina sorgente**.

### Connectivity Servers, compressione Boldi e Vigna
Se vogliamo sfruttare i link per migliorare ranking, fare clustering etc.. dobbiamo essere in grado di **accedere velocemente agli in-link e out-link di ogni pagina**.  
In questo senso ci viene in aiuto il **Connectivity Server**, che è un sistema che memorizza il grafo del Web in modo efficiente permettendo di fare query rapide legate alla connettività tra pagine.

Per ogni URL posso recuperare rapidamente, in particolare, gli in-link e out-link. Vedremo che questo è fondamentale ad esempio per PageRank, dove servono i link entranti ed uscenti per propagare l'importanza tra pagine ma anche in HITS, dove servono i link per costruire l'insieme di pagine collegate alla root set (base set). 

Se memorizzassimo ogni URL come stringa, memorizzare tutti i link sarebbe intrattabile: per questo motivo ogni URL viene tipicamente assogiato ad un intero. A quel punto il grafo può essere rappresentato con numeri invece che con stringhe URL. 

Per memorizzare il grafo si possono usare diverse strutture dati, una di queste sono le **liste di adiacenza**: per ogni nodo (pagina) posso memorizzare una lista dei nodi a cui il nodo punta. (es. URL 1 ha come lista di adiacenza [2, 3, 4] se punta a URL 2, 3 e 4).

Nonostante rappresentare interi è sicuramente meglio che rappresentare stringhe, il Web è enorme. Se ipotizziamo un Web con 4 miliardi di pagine (attualmente circa 1.3 miliardi) --> per rappresentare un nodo servirebbero 32 bit. Ogni link collega una sorgente e una destinazione, quindi ingenuamente potrei rappresentare ogni hyperlink con 64 bit. 

Chiaramente questa soluzione Naive è assolutamente intrattabile, entra in gioco quindi la **compressione di Boldi e Vigna delle liste di adiacenza**.  
Si vuole mantenere in memoria RAM le liste di adiacenza dei nodi (in quanto se ogni query sui link dovesse andare su disco gli algoritmi sarebbero troppo lenti). Per farlo è necessario trovare una compressione **lossless** molto efficiente. 

**Boldi e Vigna** permettono di compirere la rappresentazione del grafo arrivando a circa 3 bit per link, ma lavori successivi ancora più spinti arrivano a circa 2 bit per link.

**Main idea** di Boldi e Vigna: ordinare lessicograficamente tutti gli URL e sfruttare il fatto che URL vicini hanno spesso liste di adiacenza simili (vedi proprietà di Locality)

Non si spiegherà con precisione come funziona questa compressione perché è un troiaio, ma l'idea che c'è dietro, Si sfruttano **diverse proprietà del web** per comprimere le liste di adiacenza, tra cui:
- **similarity tra liste**: pagine simili o vicine nello stesso sito spesso hanno link simili, ad esempio:
    ```text 
    www.univ.it/informatica/corsi
    www.univ.it/informatica/docenti
    www.univ.it/informatica/esami
    ```
    tutte queste pagine probabilmente hanno link comuni (tipo Home, Dipartimento, Segreteria etc..) quindi l'idea è, invece di memorizzare ogni lista da 0, possiamo memorizzarne una come riferimento e rappresentare le altre come differenze rispetto a quella (es. la lista di adiacenza di www.univ.it/informatica/corsi potrebbe essere [1, 2, 3, 4], quella di www.univ.it/informatica/docenti potrebbe essere rappresentata come [1, 2, 3, 4, 5, 6] --> memorizzo la prima lista e poi per la seconda memorizzo solo i link in più, ovvero [5, 6]).
- **Locality**: se ordino lessicograficamente le pagine, allora pagine dello stesso sito finiscono vicine. Spesso per similarity pagine dello stesso sito hanno link simili --> se ordino le pagine lessicograficamente prima di assegnare l'ID, allora molti link saranno verso ID numericamente vicini --> invece di memorizzare numeri grandi, posso sfruttare tecniche di compressione per memorizzare solo le differenze e risparmiare. Empiricamente Boldi e Vigna hanno visto che in questo senso è utile lavorare a blocchi di 7 pagine.
    ```text
    www.stanford.edu/alchemy
    www.stanford.edu/biology
    www.stanford.edu/biology/plant
    www.stanford.edu/biology/plant/copyright
    www.stanford.edu/biology/plant/people
    www.stanford.edu/chemistry
    ----->
    pagina 1000 → [pagina 1003, pagina 1010, pagina 997]
    ```
- **Gap Encoding**: vabbé lo sappiamo se ho una lista ordinata di interi [10, 15, 21, 30] invece di memorizzare tutti i valori, posso memorizzare solo il primo e poi le differenze successive [10, 5, 6, 9]. Una volta passati alla forma gap, si possono applicare tecniche di compressione tipo gamma encoding o variable byte encoding per comprimere ulteriormente 

### **PageRank**
Prima di PageRank: **Citation Analysis**. Nel mondo accademico, un articolo è considerato più influente se viene citato spesso da altri articoli. In particolare però anche un articolo poco citato, ma citato da articoli molto influenti, viene considerato importante.  
In questo senso PageRank non nasce dal nulla, ma dall'idea di valutare l'importanza di un documento in base ai collegamenti ricevuti.

**Idea di PageRank**: l'idea alla base di PageRank è immaginare un utente, detto **random surfer**, che naviga casualmente nel web.  
Il comportamento è il seguente: parte da una pagina u.a.r., guarda i link uscenti presenti nella pagina e ne sceglie uno u.a.r., passa alla pagina linkata e ripete il processo all'infinito. 

La domanda che ci si fa con PageRank è: **dopo moltissimi passi, quale pagina visiterà più spesso il random surfer?** Le pagine visitate più spesso saranno quelle con PageRank più alto.

Una misura più semplice potrebbe essere il semplice conteggio nei link entranti per determinare l'importanza di una pagina, ma PageRank è più raffinato in quanto tiene conto **non solo di quanti link entranti ricevo, ma anche da chi li ricevo** (per via del fatto come detto prima che se sono linkato da una pagina molto linkata, allora è probabile per il random surfer di passare da quella pagina e quindi di visitare anche me).

L'idea base di PageRank non basta, per via dei **dead ends**: il Web contiene pagine senza link uscenti, per cui se il random surfer vi finisse non potrebbe più muoversi --> non ha più senso parlare di frequenza di visita all'infinito se ci si può bloccare.  
La soluzione al problema è il **teleporting**: ad gni istante il random surfer ha alta probabilità di seguire uno dei link uscenti (es. 90%) ma anche bassa probabilità di passare a una pagina u.a.r. (es. 10%). In questo modo anche se finisce in un dead end, ha sempre la possibilità di uscire da lì e continuare a navigare.

In realtà il teleporting, oltre al dead end, risolve anche un altro problema: **la possibilità che il random surfer resti bloccato in una zona del grafo**. Se infatti ad esempio esiste una componente fortemente connessa (es. un gruppo di pagine che si linkano tra loro ma non hanno link uscenti verso l'esterno), allora il random surfer una volta che ci capita resta bloccato lì. In questo modo invece siamo certi che all'infinito prima o poi il random surfer uscirà da quella zona e potrà visitare anche altre pagine.



#### Formalizzazione con Markov
Una catena di markov è un modello matematico che descrive un processo stocastico in cui una variabile aleatoria passa ad uno stato a un altro con una certa probabilità, che dipende solo dallo stato attuale.

Nel nostro caso gli stati sono le pagine web, e la scelta di un link rappresenta la transizione da uno stato a un altro. Il comportamento della catena è descritta dalla **matrice di transizione** $P$, dove P[i][j] rappresenta la probabilità di passare dalla pagina i alla pagina j. Nel nostro caso quindi, avendo n pagine, $P$ è una matrice $n \times n$. Ogni riga della matrice rappresenta la probabilità di passare da una pagina i a tutte le altre pagine j, e per questo deve sommare a 1.

Per inserire il teleporting nella matrice di transizione, si fa quanto segue. Se ci sono $n$ pagine e la probabilità di teleporting è $\alpha$, allora la probabilità di saltare verso una pagina specifica è $\frac{\alpha}{n}$, mentre la probabilità di seguire un link uscente è $1 - \alpha$ --> distribuisco quest'ultima probabilità sugli out-link della pagina. Per fare questa operazione si usa la seguente operazione matriciale:
$$P' = (1 - \alpha) P + \alpha \frac{1}{n} E$$
dove $E$ è una matrice $n \times n$ in cui ogni elemento è 1 (quindi $\frac{1}{n} E$ è una matrice in cui ogni elemento è $\frac{1}{n}$, ovvero la probabilità di saltare verso una pagina specifica).

es. $alpha = 0.1$, e ho una pagina con 3 link uscenti --> nella matrice di transizione P, la riga corrispondente a quella pagina avrà 0.9/3 = 0.3 per le pagine linkate e 0.1/n per tutte le pagine (compresa se stessa).

Si introduce poi il concetto di **catena ergodica** (aperiodica e irriducibile, ossia tutti raggiungibili da tutti e aperiodicità vattela a studia da Salvi): senza entrare nei dettagli, se una catena di Markov è ergodica e ha numero di nodi finito, allora nel lungo periodo esiste un'unica distribuzione stabile (detta **distribuzione stazionaria**) di probabilità sulle pagine per il random surfer, indipendente dal punto di partenza.  

Intuitivamente significa che quando il random surfer naviga per un tempo sufficientemente lungo, la probabilità di trovarsi su una pagina specifica converge a un valore fisso che non cambia più anche se il random surfer continua a navigare.

Possiamo indicare la distribuzione stazionaria come un vettore del tipo:
$$ \pi = [\pi_1, \pi_2, ..., \pi_n] $$
dove $\pi_i$ rappresenta la probabilità di trovarsi sulla pagina i. Chiaramente la somma delle $\pi_i$ deve essere 1, in quanto rappresentano una distribuzione di probabilità.

Ora consideriamo un vettore di probabilità iniziale $x$, che rappresenta al tempo 0 la probabilità per il surfer di trovarsi su ogni pagina partendo da uno stato random. Ad ogni passo chiaramente la probabilità che il surfer si trovi su un certo stato (una certa pagina) cambia, in base alla matrice di transizione $P'$. In particolare, dopo un passo, la nuova distribuzione di probabilità sarà data da:
$$ x' = x P' $$
Dopo due passi, sarà:
$$ x'' = x' P' = (x P') P' = x (P')^2 $$
Dopo $k$ passi, sarà:
$$ x^{(k)} = x (P')^k $$
Se facciamo tendere $k$ all'infinito, e se la catena è ergodica, allora $x^{(k)}$ converge alla distribuzione stazionaria $\pi$:
$$ \lim_{k \to \infty} x (P')^k = \pi $$
per la quale, essendo stazionaria, varrà che:
$$ \pi P' = \pi $$
e quindi la probabilità che il random surfer si trovi su una data pagina, a quel punto, resta la stessa indipendentemente da quanti passi fa.

Nel contesto del Pagerank, **$a_i$ rappresenterà proprio il PageRank della pagina $i$**, quindi le pagine con valore $a_i$ più alto saranno proprio quelle più importanti secondo PageRank.

Un altro modo per interpretare pagerank è algebricamente: se guardiamo la formula $\pi P' = \pi$, allora $\pi$ è un autovettore di $P'$ con autovalore 1. Poiché per una matrice di transizione il massimo autovalore è 1, allora $\pi$ è proprio l'autovettore principale di $P'$. 

**Come calcolare PageRank in pratica?**  
In teoria secondo Markov basterebbe risolvere il sistema di equazioni $\pi P' = \pi$ con anche l'$n+1$-esima equazione che impone che la somma di $\pi_i$ sia 1. In pratica però sul Web la matrice è enorme e assolutamente intrattabile.

In pratica quindi si usa un algoritmo iterativo detto **Power Iteration**: si parte da un vettore di probabilità iniziale $x$, spesso uniforme, e poi si ripetono le operazioni:
```text
x¹ = x⁰P
x² = x¹P
x³ = x²P
...
```
finché $x^k$ non arriva abbastanza vicino alla convergenza, ovvero finché la differenza tra $x^k$ e $x^{k-1}$ è minore di una certa soglia $\epsilon$. 

Esercizio:

<img src="img/444.png" width="400"/>

<img src="img/555.png" width="400"/>

Dove nell'ultima immagine ad esempio per passare da 1/2 a 7/14 l'operazione è semplicemente:
$$ 0.9 \cdot \text{vecchio valore in P} + 0.1 \cdot \frac{1}{n} = 0.9 \cdot 1/2 + 0.1 \cdot \frac{1}{6} $$

### HITS
HITS sta per Hyperlink-Induced Topic Search, l'**idea** di base è la seguente.  
Quando si fa una query ampia a livello di topic, ad esempio "machine learning" o "java", allora l'utente non vuole necessariamente una pagina migliore, ma si potrebbe voler distinguere tra due tipi di pagine:
1. **Pagine che sono buone fonti autorevoli sull'argomento**
2. **Pagine che sono buone raccolte di link verso pagine autorevoli**

Queste due categorie di pagine vengono chiamate rispettivamente **Authorities** e **Hubs**.  
In generale una buona hub punta a molte buone authorities, mentre una buona authority è puntata da molti buoni hub. Questa è una definizione circolare che l'algoritmo può risolvere iterativamente. 

es. supponiamo di avere:
```text
H1 → A1
H1 → A2
H1 → A3

H2 → A1
H2 → A2
```
Se H1 e H2 sono buoni hub, allora A1 e A2 ricevono conferma come authorities perché linkate da più hub. Allo stesso tempo se A1, A2 e A3 sono buone authorities, allora H1 e H2 ricevono conferma come hub perché linkano molte authorities.

La differenza più grande tra HITS e PageRank è che PageRank, almeno nella versione base, è del tutto **query-independent**: si misura l'importanza di una pagina a prescindere dalla query dell'utente.  
Al contrario HITS è **query-dependent** dal momento che parte da una query testuale, costruisce un sottinsieme di pagine rilevanti per quella query e poi calcola hub e authorities dentro quel sottografo.

In particolare HITS funziona meglio per **broad topic queries**, ossia query ampie a livello di argomento, come visto negli esempi di prima. Al contrario ovviamente se la query è troppo specifica tipicamente è perché l'utente sta cercando proprio una certa pagina, e quindi in quel caso hub probabilmente manco ce ne sono.

#### Algoritmo HITS
HITS funziona in due fasi principali:
1. **Costruzione di un insieme di pagine candidate** (extract a **base set**)
2. **Calcolo iterativo dei punteggi hub e authority**

Immaginiamo che l'utente faccia la query "browser". Quello che si fa allora è sfruttare anzitutto un motore di ricerca testuale classico, recuperando diverse pagine candidate che contengano il termine browser. Questo primo insieme di pagine è detto **root set**. 

Tuttavia il root set da solo può essere troppo limitato, perché pagine autorevoli potrebbero non contenere esattamente la parola "browser" ma essere comunque rilevanti. Inoltre per capire chi sono gli hub dobbiamo guardare anche le pagine a cui puntano le pagine del root set.

Per questo motivo HITS espande il root set: si aggiunge al root set ogni pagina che **punta a una pagina del root set** (in-link) e ogni pagina che **viene puntata da una pagina del root set** (out-link). Si ottiene così il base set:
```text
base set = root set + in-neighbors + out-neighbors
```
Per ottenere questi out-neighbors e in-neighbors è fondamentale avere un **connectivity server** che ci permetta di accedere velocemente a queste informazioni sui link.

Una volta costruito il base set, si assegnano ad ogni pagina due punteggi:
- $h(x) = $ hub score della pagina x
- $a(x) = $ authority score della pagina x

Inizialmente si assegna $h(x) = 1$ e $a(x) = 1$ per ogni pagina x del base set.  

Dopodiché il punteggio hub di una pagina $x$ si ottiene sommando gli authority score delle pagine a cui $x$ punta (infatti una pagina è buon hub se punta a molte buone authorities):
$$ h(x) = \sum_{y: x \to y} a(y) $$

Per quel che riguarda il punteggio authority di una pagina $x$ invece, questo si ottiene sommando gli hub score delle pagine che puntano a $x$ (infatti una pagina è buona authority se è puntata da molti buoni hub):
$$ a(x) = \sum_{y: y \to x} h(y) $$
Capiamo quindi che perché questo algoritmo funzioni è fondamentale **iterare**: inizialmente tutti i punteggi sono 1, ma dopo la prima iterazione i punteggi hub e authority cambiano, e quindi è necessario ricalcolarli più volte finché non convergono (ovvero finché la differenza tra i punteggi di un'iterazione e quelli dell'iterazione precedente è minore di una certa soglia $\epsilon$). Empiricamente, HITS si avvicina sufficientemente alla stabilità entro 5 iterazioni.

Ad ogni iterazione, è importante **normalizzare i punteggi**. Questo perché continuando a sommare i punteggi iterazione dopo iterazione, questi valori diventano enormi. Tuttavia a noi interessano solo valori relativi (es. non ci interessa se una pagina ha score 100 e un altra 50, li posso vedere allo stesso modo normalizzati come 1 e 0.5, il rapporto è lo stesso).

Per questi motivi dopo ogni iterazione i vettori $\vec{h}$ e $\vec{a}$ vengono normalizzati, ad esempio dividendo ogni elemento per la norma del vettore (es. $h(x) = \frac{h(x)}{\sqrt{\sum_{x} h(x)^2}}$).



#### Forma matriciale e convergenza 
Si passa a una dimostrazione alla cazzo di cane della convergenza di HITS usando la matrice di adiacenza (per vederla fatta bene segui il corso di Analisi di Reti).

Supponiamo che il base set abbia $n$ pagine. Costruiamo quindi una matrice $A$ di dimensione $n \times n$, dove $A[i][j] = 1$ se la pagina i punta alla pagina j, altrimenti $A[i][j] = 0$. Questa rappresenta proprio la matrice di adiacenza del sottografo fatto dalle pagine del base set.

Siano come anticipato $\vec{h}$ e $\vec{a}$ i vettori dei punteggi hub e authority, allora possiamo riscrivere le formule iterative di aggiornamento in forma matriciale:
$$ \vec{h} = A \vec{a} \qquad \vec{a} = A^T \vec{h} $$
Dove la prima intuitivamente ha senso perché il punteggio hub di una pagina è la somma delle authority delle pagine a cui punta --> la riga $i$ della matrice dice proprio quali pagine sono puntate dalla pagina $i$ --> moltiplicando $A$ per $\vec{a}$ per ogni pagina $i$ ottengo proprio la somma delle authority delle pagine a cui punta $i$.

La seconda intuitivamente ha senso perché il punteggio authority di una pagina è la somma degli hub delle pagine che puntano a quella pagina --> la colonna $j$ della matrice dice proprio quali pagine puntano alla pagina $j$ (perché ho 1 se la pagina $i$ punta alla pagina $j$) --> faccio la trasposta per vedere le colonne come righe e moltiplico per $\vec{h}$ per ottenere la somma degli hub delle pagine che puntano a $j$.

Ci ricolleghiamo ora agli autovettori sostituendo una formula nell'altra:
$$ \vec{h} = A \vec{a} = A (A^T \vec{h}) = (A A^T) \vec{h} $$
$$ \vec{a} = A^T \vec{h} = A^T (A \vec{a}) = (A^T A) \vec{a} $$
Quindi HITS sta di fatto iterativamente calcolando, con normalizzazione ad ogni passo, l'autovettore principale di $A A^T$ per i punteggi hub e l'autovettore principale di $A^T A$ per i punteggi authority.

L'algoritmo in questo senso è una forma di power iteration per calcolare autovettori. Per la dimostrazione della convergenza, ci si ricollega al fatto che la power iteration converge all'autovettore principale (ossia quello associato ad autovalore più grande) della matrice sotto certe condizioni.


#### Problemi di HITS
Alcuni dei principali problemi di HITS sono i seguenti:
- **Topic drift**: avviene quando l'algoritmo si allontana dal topic originale. Ad esempio potrei partire da una query specifica, ma espandendo il root set con link entranti e uscenti potrei finire in una zona del grafo con argomento più generale o addirittura diverso. Sempre classico esempio jaguar car, in questo caso l'algoritmo potrebbe dare molta importanza a pagine autorevoli su auto in generale, non specificatamente jaguar.
- **Mutually reinforcing affilates**: se un gruppo di pagine appartenenti alla stessa organizzazione si linka molto internamente, può aumentare artificialmente i punteggi hub e authority